# Atividade 3 - Chatbot baseado em dados estruturados (JSON) com apoio de LLM

Arquitetura do pipeline:

1. Ingestão: Carregamento de arquivos JSON (Matriz, Horários e Calendário) do Google Drive.

2.  Transformação: Conversão de estruturas aninhadas em "frases de contexto" ricas em metadados.

3.  Indexação: Geração de vetores via `text-embedding-3-small` e armazenamento em base local FAISS.

4.  Recuperação: Busca semântica por similaridade de cosseno para encontrar os fragmentos mais relevantes.

5.  Geração: Síntese da resposta final utilizando o modelo `gpt-4o-mini`, com instruções rigorosas de não inventar dados fora do contexto fornecido.

<br>

Para rodar os exercícios você precisará fornecer o caminho dos arquivos JSON na variável `PATH_JSONS = "caminho/arquivos` na Seção 5:

* Calendário acadêmico (`calendario-2026.json`);  
* Horários das disciplinas (`dsm-horario-2026-1.json`);  
* Matriz curricular (`dsm-matriz-curricular.json`).


In [ ]:
# @title 1.1. Intalação e importações
!pip install -q openai faiss-cpu tiktoken

import os
import json
import pandas as pd
import numpy as np
import faiss
import pickle
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any
from google.colab import drive, userdata
from openai import OpenAI
from google.colab.userdata import SecretNotFoundError, NotebookAccessError

In [ ]:
# @title 1.2. Configurações do RAG e chave OpenAI

def load_openai_key_from_colab(key_name:str):
  try:
      key = userdata.get(key_name)
      os.environ[key_name] = key
      print(f"✅ {key_name} carregada com sucesso a partir do Colab Secrets.")
  except (SecretNotFoundError, NotebookAccessError):
      print(f"⚠️ {key_name} não configurada corretamente no Colab Secrets.")

@dataclass
class RAGConfig:
    embed_model: str = "text-embedding-3-small"
    gen_model: str = "gpt-4o-mini"
    temperature: float = 0.0
    top_k: int = 15
    max_context_tokens: int = 3500

# Inicialização do estado
class RAGState:
    def __init__(self, config: RAGConfig):
        self.config = config
        load_openai_key_from_colab('OPENAI_API_KEY')
        self.client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
        self.documents = []
        self.vector_store = None

In [ ]:
# @title 1.3. Integração com Google Drive

def load_json_files_from_drive(drive_root, relative_path: str) -> List[Dict]:
    """
    Monta o drive e carrega todos os arquivos .json de uma pasta específica.
    """
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # Caminho absoluto baseado na pasta informada
    full_path = f"{drive_root}/{relative_path}"

    if not os.path.isdir(full_path):
        print(f"Erro: A pasta {full_path} não foi encontrada no seu Drive.")
        return []

    all_json_data = []
    print(f"Lendo arquivos de: {full_path}...")

    for filename in os.listdir(full_path):
        if filename.endswith(".json"):
            file_path = os.path.join(full_path, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = json.load(f)
                    all_json_data.append({
                        "metadata": {"file_name": filename},
                        "content": content
                    })
                print(f"✓ Carregado: {filename}")
            except Exception as e:
                print(f"✗ Erro ao carregar {filename}: {e}")

    return all_json_data

In [ ]:
# @title 2. Processamento Estruturado de JSON
def jsons_to_docs(json_data: List[Dict]) -> List[Dict]:
    """
    Transforma JSONs brutos em documentos de texto enriquecidos com metadados.
    """
    docs = []

    for item in json_data:
        file_name = item.get("metadata", {}).get("file_name", "desconhecido")
        content = item.get("content", {})

        # Identifica o curso/domínio
        curso = content.get("sigla", content.get("curso", "Geral"))

        # Estratégia de 'Flattening' por tipo de documento
        if "matriz" in content:
            for sem in content["matriz"]:
                for disc in sem["disciplinas"]:
                    text = (f"Curso: {curso} | Semestre: {sem['semestre']}º | "
                            f"Disciplina: {disc['nome']} | Carga Horária: {disc['carga_horaria']}h | "
                            f"Modalidade: {disc['modalidade']}")
                    docs.append({"text": text, "metadata": {"source": file_name, "type": "matriz"}})

        elif "semestres" in content: # Horários
            for sem in content["semestres"]:
                for disc in sem["disciplinas"]:
                    horarios = ", ".join(disc["horarios"])
                    text = (f"Horário {curso} | Semestre: {sem['semestre']}º | "
                            f"Disciplina: {disc['disciplina']} | Professor: {disc['professor']} | "
                            f"Horários: {horarios}")
                    docs.append({"text": text, "metadata": {"source": file_name, "type": "horario"}})

        elif "eventos" in content: # Calendário
            for ev in content["eventos"]:
                text = f"Calendário Acadêmico {content.get('ano', '')} | Evento: {ev['nome']} | Data: {ev['inicio']} até {ev['fim']}"
                docs.append({"text": text, "metadata": {"source": file_name, "type": "calendario"}})

    return docs

In [ ]:
# @title 3. Implementação Completa do Pipeline RAG Estruturado

def get_embeddings(texts: List[str], client: OpenAI, model: str):
    res = client.embeddings.create(input=texts, model=model)
    return [d.embedding for d in res.data]

def build_vector_store(state: RAGState, docs: List[Dict]):
    """Cria o índice FAISS a partir dos documentos processados."""
    texts = [d["text"] for d in docs]
    embeddings = get_embeddings(texts, state.client, state.config.embed_model)

    # Converte para float32 para o FAISS
    dimension = len(embeddings[0])
    embeddings_np = np.array(embeddings).astype('float32')

    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings_np)

    state.vector_store = index
    state.documents = docs # Mantém a referência dos textos originais
    print(f"Índice construído com {len(docs)} fragmentos estruturados.")

def search_docs(state: RAGState, query: str, k: int = 10) -> str:
    """Busca os fragmentos mais relevantes no FAISS."""
    q_embed = get_embeddings([query], state.client, state.config.embed_model)
    q_embed_np = np.array(q_embed).astype('float32')

    distances, indices = state.vector_store.search(q_embed_np, k)

    relevant_chunks = []
    for idx in indices[0]:
        if idx != -1:
            relevant_chunks.append(state.documents[idx]["text"])

    return "\n".join(relevant_chunks)


In [ ]:
# @title 4. Persistência do Índice e Execução no Drive

def save_vector_store(state: RAGState, folder_path: str):
    """Salva o índice FAISS e os metadados dos documentos no Drive."""
    index_path = os.path.join(folder_path, "base/index.faiss")
    docs_path = os.path.join(folder_path, "base/documents.pkl")

    # Ensure the directory exists
    os.makedirs(os.path.dirname(index_path), exist_ok=True)

    # Salva o índice FAISS
    faiss.write_index(state.vector_store, index_path)

    # Salva os documentos (textos e metadados) usando pickle
    with open(docs_path, "wb") as f:
        pickle.dump(state.documents, f)

    print(f"✓ Índice e documentos salvos com sucesso em: {folder_path}/base")

def load_vector_store(state: RAGState, folder_path: str) -> bool:
    """Tenta carregar o índice e documentos do Drive."""
    index_path = os.path.join(folder_path, "base/index.faiss")
    docs_path = os.path.join(folder_path, "base/documents.pkl")

    if os.path.exists(index_path) and os.path.exists(docs_path):
        state.vector_store = faiss.read_index(index_path)
        with open(docs_path, "rb") as f:
            state.documents = pickle.load(f)
        print("✓ Índice carregado do Google Drive (cache).")
        return True
    return False

def build_vector_store(state: RAGState, docs: List[Dict]):
    """Cria o índice FAISS para busca semântica nos dados estruturados."""
    if not docs:
        return

    texts = [d["text"] for d in docs]
    # Reutiliza a função de embeddings definida anteriormente
    embeddings = get_embeddings(texts, state.client, state.config.embed_model)

    dimension = len(embeddings[0])
    embeddings_np = np.array(embeddings).astype('float32')

    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings_np)

    state.vector_store = index
    state.documents = docs
    print(f"Índice FAISS construído com {len(docs)} documentos.")

def answer(state: RAGState, question: str) -> str:
    """Busca contexto e gera resposta usando o GPT-4o-mini."""
    # 1. Busca no FAISS
    q_embed = get_embeddings([question], state.client, state.config.embed_model)
    q_embed_np = np.array(q_embed).astype('float32')
    _, indices = state.vector_store.search(q_embed_np, state.config.top_k)

    context_chunks = []
    for idx in indices[0]:
        if idx != -1:
            context_chunks.append(state.documents[idx]["text"])

    context_text = "\n".join(context_chunks)

    # 2. Geração da Resposta
    system_prompt = (
        "Você é o assistente virtual da Secretaria da Fatec Jacareí.\n"
        "Sua base de dados é composta por JSONs estruturados de horários, matrizes e calendário.\n"
        "REGRAS:\n"
        "- Seja preciso. Se a pergunta é sobre carga horária, use os dados da 'Matriz'.\n"
        "- Se é sobre aulas ou professores, use 'Horário'.\n"
        "- Se a pergunta envolver professores e disciplinas use os dados de 'Horário' e 'Matriz'.\n"
        "- Se a pergunta envolver cálculo total de horas use os dados de 'Matriz'.\n"
        "- Se não encontrar a informação, diga que não consta nos registros oficiais."
    )
     # Exercício 1 - Retire o comentário para ver o contexto na tela
    print(f"Contexto:\n{context_text}")
    response = state.client.chat.completions.create(
        model=state.config.gen_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Contexto:\n{context_text}\n\nPergunta: {question}"}
        ],
        temperature=state.config.temperature
    )
    return response.choices[0].message.content

In [ ]:
# @title 5. Execução do Fluxo Completo

# 1. Configurações
PATH_JSONS = "fatec/jacarei/DSM/PLN/atividades/Atividade 3/arquivos"
config = RAGConfig()
state = RAGState(config)

# 2. Montar Drive e definir caminho absoluto
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
full_path = f"/content/drive/MyDrive/{PATH_JSONS}"

# 3. Tentar carregar índice existente, senão constrói um novo
if not load_vector_store(state, full_path):
    print("Índice não encontrado. Iniciando processamento dos JSONs...")

    dados_brutos = load_json_files_from_drive("/content/drive/MyDrive/", PATH_JSONS)
    if dados_brutos:
        documentos = jsons_to_docs(dados_brutos)
        print("docs:", documentos)
        build_vector_store(state, documentos)

        # Salva para a próxima vez
        save_vector_store(state, full_path)
    else:
        print("Erro: Nenhum arquivo JSON encontrado para indexar.")


In [ ]:
# @title 6. Interface de Chat
while True:
  pergunta = input("\nPergunta (ou 'sair'): ").strip()
  if not pergunta or pergunta.lower() == "sair":
    break
  print(f"\nR: {answer(state, pergunta)}")